In [2]:
import pandas as pd
import numpy as np

fy2018 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_Disclosure_Data_FY2018_EOY.xlsx"

df = pd.read_excel(fy2018)

print(df.shape)
print(df.head())

(654360, 52)
          CASE_NUMBER CASE_STATUS CASE_SUBMITTED DECISION_DATE VISA_CLASS  \
0  I-200-18026-338377   CERTIFIED     2018-01-29    2018-02-02       H-1B   
1  I-200-17296-353451   CERTIFIED     2017-10-23    2017-10-27       H-1B   
2  I-200-18242-524477   CERTIFIED     2018-08-30    2018-09-06       H-1B   
3  I-200-18070-575236   CERTIFIED            NaT    2018-03-30       H-1B   
4  I-200-18243-850522   CERTIFIED     2018-08-31    2018-09-07       H-1B   

  EMPLOYMENT_START_DATE EMPLOYMENT_END_DATE                EMPLOYER_NAME  \
0            2018-07-28          2021-07-27        MICROSOFT CORPORATION   
1            2017-11-06          2020-11-06       ERNST & YOUNG U.S. LLP   
2            2018-09-10          2021-09-09                 LOGIXHUB LLC   
3            2018-09-10          2021-09-09  HEXAWARE TECHNOLOGIES, INC.   
4            2018-09-07          2021-09-06             ECLOUD LABS,INC.   

  EMPLOYER_BUSINESS_DBA       EMPLOYER_ADDRESS  ... H1B_DEPENDENT  

In [4]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'CASE_SUBMITTED', 'DECISION_DATE', 'VISA_CLASS', 'EMPLOYMENT_START_DATE', 'EMPLOYMENT_END_DATE', 'EMPLOYER_NAME', 'EMPLOYER_BUSINESS_DBA', 'EMPLOYER_ADDRESS', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_NAME', 'AGENT_ATTORNEY_CITY', 'AGENT_ATTORNEY_STATE', 'JOB_TITLE', 'SOC_CODE', 'SOC_NAME', 'NAICS_CODE', 'TOTAL_WORKERS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMP', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'FULL_TIME_POSITION', 'PREVAILING_WAGE', 'PW_UNIT_OF_PAY', 'PW_WAGE_LEVEL', 'PW_SOURCE', 'PW_SOURCE_YEAR', 'PW_SOURCE_OTHER', 'WAGE_RATE_OF_PAY_FROM', 'WAGE_RATE_OF_PAY_TO', 'WAGE_UNIT_OF_PAY', 'H1B_DEPENDENT', 'WILLFUL_VIOLATOR', 'SUPPORT_H1B', 'LABOR_CON_AGREE', 'PUBLIC_DISCLOSURE_LOCATION', 'WORKSITE_CITY', 'WORKSITE_COUNTY', 'WORKSITE_STATE', 'WORKSITE_POSTAL_CO

In [6]:
import pandas as pd

df = pd.read_excel(fy2018)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
GOOGLE LLC                                                              6099
MICROSOFT CORPORATION                                                   5552
AMAZON.COM SERVICES, INC.                                               3894
FACEBOOK, INC.                                                          2731
APPLE INC.                                                              1964
TESLA, INC.                                                             1077
AMAZON WEB SERVICES, INC.                                               1008
AMAZON FULFILLMENT SERVICES, INC.                                        797
NVIDIA CORPORATION                                                       528
GOOGLE INC.                                                              478
MICRON TECHNOLOGY, INC.                                                  354
ADVANCED MICRO DEVICES, INC.                                             334
MICRONET IT SOLUTIONS, INC.                                   

In [7]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Alphabet     6099
Microsoft    5552
Meta         2731
Apple        1964
Tesla        1077
NVIDIA        528
Micron        354
AMD           334
Name: count, dtype: int64


In [8]:
df_clean = df.copy()

In [11]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
New York    713
Seattle     524
Austin      439
Dallas       15
San Jose      2
Name: count, dtype: int64


In [15]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year    1690
Hour       3
Name: count, dtype: int64


In [17]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

      Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  Annual_Wage
79        AMD    Austin               61523.00             YEAR     61523.00
126       AMD    Austin               90801.00             YEAR     90801.00
312  Alphabet  New York              164000.00             YEAR    164000.00
331      Meta   Seattle              155000.00             YEAR    155000.00
688       AMD    Austin              131972.00             YEAR    131972.00
712      Meta   Seattle              145336.47             YEAR    145336.47
852       AMD    Austin               94305.00             YEAR     94305.00
938  Alphabet   Seattle              139000.00             YEAR    139000.00
948      Meta  New York              170000.00             YEAR    170000.00
953  Alphabet   Seattle              119000.00             YEAR    119000.00


In [19]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    1.693000e+03
mean     3.144792e+05
std      7.326732e+06
min      5.755400e+04
25%      1.152450e+05
50%      1.350000e+05
75%      1.544000e+05
max      3.016000e+08
Name: Annual_Wage, dtype: float64


In [21]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [23]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage     Mean_Wage
0     Austin        AMD                139     100755.0  1.057475e+05
1     Austin   Alphabet                 16     101500.0  1.062463e+05
2     Austin      Apple                197     115000.0  1.131431e+05
3     Austin       Meta                 45     108000.0  1.158554e+05
4     Austin     Micron                  4     119717.5  1.164088e+05
5     Austin  Microsoft                  1     147430.0  1.474300e+05
6     Austin     NVIDIA                 31     121160.0  1.240628e+05
7     Austin      Tesla                  6     125000.0  1.330000e+05
8     Dallas        AMD                  1     106891.0  1.068910e+05
9     Dallas      Apple                 11     110261.0  1.119439e+05
10    Dallas       Meta                  2     140500.0  1.405000e+05
11    Dallas  Microsoft                  1     139000.0  1.390000e+05
12  New York   Alphabet                469     138000.0  1.416486e+05
13  New York      Ap

In [27]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_NAME",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2018.csv",
    index=False
)